# 🤖 Transformer Word Predictor — Train GPT-2 on OpenWebText

Fine-tunes GPT-2 small (124M) on **1% of OpenWebText (~100M tokens)**.

### Features
- **Google Drive checkpointing** — progress is saved automatically
- **Auto-resume** — re-run the training cell after a disconnect
- **Checkpoints every 500 steps** — minimal data loss on interruption

**Runtime**: `Runtime → Change runtime type → T4 GPU`

In [ ]:
# ============================================================
# 1. SETUP & MOUNT DRIVE
# ============================================================
!pip install -q torch transformers datasets

import torch
print(f'PyTorch {torch.__version__} | CUDA {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_BASE = '/content/drive/MyDrive/Wordprediktor'
DRIVE_CHECKPOINT_DIR = os.path.join(DRIVE_BASE, 'checkpoints-transformer')
DRIVE_MODEL_DIR = os.path.join(DRIVE_BASE, 'models', 'gpt2-openwebtext')
os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)
os.makedirs(DRIVE_MODEL_DIR, exist_ok=True)
print(f'Checkpoint dir: {DRIVE_CHECKPOINT_DIR}')
print(f'Model dir:      {DRIVE_MODEL_DIR}')

In [ ]:
# ============================================================
# 2. LOAD & TOKENIZE OPENWEBTEXT (1% ≈ 100M tokens)
# ============================================================
import os, math
from datasets import load_dataset, DatasetDict
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    DataCollatorForLanguageModeling, Trainer, TrainingArguments,
)

MODEL_NAME = 'gpt2'
OUTPUT_DIR = DRIVE_CHECKPOINT_DIR
BLOCK_SIZE = 256
DATA_PCT = 1  # 1% of OpenWebText ≈ 100M tokens

print(f'Loading OpenWebText ({DATA_PCT}% subset) ...')
full = load_dataset('Skylion007/openwebtext', split=f'train[:{DATA_PCT}%]')
# OpenWebText has no validation split — carve out 5%
split = full.train_test_split(test_size=0.05, seed=42)
raw = DatasetDict({'train': split['train'], 'validation': split['test']})
print(f"Train: {len(raw['train']):,}   Val: {len(raw['validation']):,} documents")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

def tokenize_fn(ex):
    return tokenizer(ex['text'], truncation=False, return_attention_mask=False)

print('Tokenizing ...')
tokenized = raw.map(tokenize_fn, batched=True, remove_columns=['text'], desc='Tokenizing')

def group_texts(ex):
    concat = {k: sum(ex[k], []) for k in ex}
    n = (len(concat['input_ids']) // BLOCK_SIZE) * BLOCK_SIZE
    res = {k: [v[i:i+BLOCK_SIZE] for i in range(0, n, BLOCK_SIZE)] for k, v in concat.items()}
    res['labels'] = res['input_ids'].copy()
    return res

print(f'Grouping into blocks of {BLOCK_SIZE} tokens ...')
lm_dataset = tokenized.map(group_texts, batched=True, desc='Grouping')
print(f"Train blocks: {len(lm_dataset['train']):,}   Val blocks: {len(lm_dataset['validation']):,}")

In [ ]:
# ============================================================
# 3. TRAIN (with auto-resume from Drive checkpoints)
# ============================================================
EPOCHS = 3
BATCH_SIZE = 8
LEARNING_RATE = 2e-5
WARMUP_STEPS = 500
SAVE_STEPS = 500
EVAL_STEPS = 500
LOGGING_STEPS = 100

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.resize_token_embeddings(len(tokenizer))

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    weight_decay=0.01,
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=3,
    eval_strategy='steps',
    eval_steps=EVAL_STEPS,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    fp16=torch.cuda.is_available(),
    report_to='none',
    dataloader_num_workers=2,
    save_safetensors=True,
)

trainer = Trainer(
    model=model, args=training_args,
    train_dataset=lm_dataset['train'],
    eval_dataset=lm_dataset['validation'],
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)

# ---- Auto-resume from latest checkpoint ----
resume_ckpt = None
if os.path.isdir(OUTPUT_DIR):
    ckpts = [
        os.path.join(OUTPUT_DIR, d)
        for d in os.listdir(OUTPUT_DIR)
        if d.startswith('checkpoint-')
    ]
    if ckpts:
        resume_ckpt = max(ckpts, key=os.path.getmtime)
        print(f'🔄 Resuming from: {resume_ckpt}')
    else:
        print('🆕 No checkpoint found — starting fresh')

print('\nTraining ...')
trainer.train(resume_from_checkpoint=resume_ckpt)

In [ ]:
# ============================================================
# 4. EVALUATE & SAVE FINAL MODEL
# ============================================================
eval_result = trainer.evaluate()
ppl = math.exp(eval_result['eval_loss'])
print(f'Validation perplexity: {ppl:.2f}')

trainer.save_model(DRIVE_MODEL_DIR)
tokenizer.save_pretrained(DRIVE_MODEL_DIR)
print(f'\nFinal model saved to: {DRIVE_MODEL_DIR} ✓')

In [ ]:
# ============================================================
# 5. TRANSFORMER PREDICTOR
# ============================================================
import torch
from typing import List, Tuple, Optional
from transformers import AutoTokenizer, AutoModelForCausalLM


class TransformerPredictor:
    """Word-completion predictor using GPT-2 with beam search over BPE tokens."""

    def __init__(self, model_path='gpt2', device=None, max_steps=10):
        self.device = device or (
            'cuda' if torch.cuda.is_available()
            else 'mps' if torch.backends.mps.is_available()
            else 'cpu'
        )
        print(f'[Predictor] Loading {model_path} on {self.device}')
        self.tokenizer = AutoTokenizer.from_pretrained(model_path)
        self.tokenizer.pad_token = self.tokenizer.eos_token
        self.model = AutoModelForCausalLM.from_pretrained(model_path)
        self.model.to(self.device)
        self.model.eval()
        self.max_steps = max_steps
        self.eos_id = self.tokenizer.eos_token_id

    def predict(self, context, prefix='', top_k=5, beam_width=20):
        if not context and not prefix:
            return []
        full = context
        if prefix:
            if full and not full.endswith(' '):
                full += ' '
            full += prefix
        input_ids = self.tokenizer.encode(full, return_tensors='pt').to(self.device)
        completions = self._beam_search(input_ids, prefix, beam_width)
        seen = set()
        unique = []
        for w, s in completions:
            k = w.lower()
            if k not in seen and k:
                seen.add(k)
                unique.append((w, s))
        unique.sort(key=lambda x: x[1], reverse=True)
        return [w for w, _ in unique[:top_k]]

    def predict_from_text(self, typed_text, top_k=5):
        if not typed_text:
            return []
        if typed_text.endswith(' '):
            return self.predict(typed_text, '', top_k=top_k)
        parts = typed_text.rsplit(' ', 1)
        if len(parts) == 1:
            return self.predict('', parts[0], top_k=top_k)
        return self.predict(parts[0] + ' ', parts[1], top_k=top_k)

    def _beam_search(self, input_ids, prefix, beam_width):
        active = [(0.0, [])]
        completed = []
        pl = prefix.lower()

        for _ in range(self.max_steps):
            if not active:
                break
            cands = []
            for lp, app in active:
                if app:
                    ext = torch.tensor([app], device=self.device)
                    fids = torch.cat([input_ids, ext], dim=-1)
                else:
                    fids = input_ids
                with torch.no_grad():
                    logits = self.model(fids).logits[0, -1, :]
                log_p = torch.log_softmax(logits, dim=-1)
                vals, idxs = torch.topk(log_p, k=min(beam_width*3, log_p.size(0)))
                for i in range(vals.size(0)):
                    tid = idxs[i].item()
                    nlp = lp + vals[i].item()
                    napp = app + [tid]
                    if tid == self.eos_id:
                        w = self._word(napp)
                        if w and w.lower().startswith(pl):
                            completed.append((w, nlp))
                        continue
                    raw = self.tokenizer.decode(napp, skip_special_tokens=True)
                    text = raw.lstrip()
                    if not text:
                        cands.append((nlp, napp))
                        continue
                    if ' ' in text or '\n' in text or '\t' in text:
                        w = self._clean(text.split()[0])
                        if w and w.lower().startswith(pl):
                            completed.append((w, nlp))
                    else:
                        p = self._clean(text)
                        if not pl or p.lower().startswith(pl):
                            cands.append((nlp, napp))
            cands.sort(key=lambda x: x[0], reverse=True)
            active = cands[:beam_width]

        for lp, app in active:
            w = self._word(app)
            if w and w.lower().startswith(pl):
                completed.append((w, lp))
        return completed

    @staticmethod
    def _clean(t):
        o = []
        for c in t:
            if c.isalpha() or c in \"-'\":
                o.append(c)
            else:
                break
        return ''.join(o)

    def _word(self, ids):
        if not ids:
            return ''
        t = self.tokenizer.decode(ids, skip_special_tokens=True).lstrip()
        return self._clean(t.split()[0]) if t.split() else self._clean(t)

print('TransformerPredictor defined ✓')

In [ ]:
# ============================================================
# 6. TEST PREDICTIONS
# ============================================================
predictor = TransformerPredictor(DRIVE_MODEL_DIR)

tests = [
    'The quick brown ',
    'The quick br',
    'Machine learning is a ',
    'I went to the ',
    'pyt',
    'Artificial intelli',
    'The president of the ',
    'In the year ',
    'I want to ',
    'She said that ',
]

for t in tests:
    s = predictor.predict_from_text(t, top_k=5)
    print(f"  '{t}'  ->  {s}")

In [ ]:
# ============================================================
# 7. INTERACTIVE TEST
# ============================================================
while True:
    text = input('\nType (or quit): ')
    if text.lower() == 'quit':
        break
    print(f'  -> {predictor.predict_from_text(text, top_k=5)}')